# Activity 1: Vision AI, Two Clouds, One Structured Dataset

**Week 6 Day 4 | AWS Rekognition vs. GCP Vision on real vehicle photos**

**Estimated time:** 75 minutes
**Difficulty:** Beginner to Intermediate
**Format:** Individual
**Prerequisites:** [Activity 0](./Activity_0_Environment_and_API_Setup.md) complete, with `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`, and `GCP_API_KEY` in your repo-root `.env`

## The job

You have a folder of vehicle photos. Somebody wants a table:

| image | state | make |
| :--- | :--- | :--- |
| `1c00790f8e592ab2.jpg` | Virginia | Not Found |
| `27801b00dbefaa6d.jpg` | Pennsylvania | Toyota |

That is the whole assignment. Pixels in, rows out. Nobody wants the photos and nobody wants a wall of printed text. They want a dataset they can query.

Two managed AI services can read the text in those photos for you. Neither one gives you a `state` column. They give you raw text, and the gap between "raw text" and "a column" is the part you write.

## How this notebook works

Every idea appears three times:

1. **Explained** in a markdown cell, before you run anything.
2. **Worked once**, on a single photo, in small cells you can inspect.
3. **Handed to you** as a `TODO` for the other 30 photos.

The cells marked `TODO` are where you write code. Everything else runs as-is. Do not skip ahead to the `TODO` cells, the worked example above each one is the answer key for the shape of the code.

## What you will learn

- That a managed AI service is a menu of operations, not one function, and how to see that menu
- What an API call actually returns, and how to navigate an unfamiliar response object
- Why the documentation, not the code, is where you find limits and options
- The difference between `LINE` and `WORD` in OCR output, and why it changes your answer
- How to turn free text into a controlled, structured field
- How to run the same job on two clouds and compare them honestly

---
## Setup

Run this first. If either line prints `False`, go back to [Activity 0](./Activity_0_Environment_and_API_Setup.md) before continuing.

In [ ]:
import os
import glob
import json
import base64

import boto3
import requests
import pandas as pd
from dotenv import load_dotenv
from PIL import Image

load_dotenv()

print("AWS credentials loaded:", "AWS_ACCESS_KEY_ID" in os.environ)
print("GCP_API_KEY loaded:    ", "GCP_API_KEY" in os.environ)

---
# 1. Rekognition is a service, not a function

Before the first call, a mental model correction that saves people a lot of confusion.

"AWS Rekognition" is not a function that reads text. It is a **service** with a menu of pre-trained models behind it. Reading text is one item on that menu. Ask for a different item, get a completely different kind of answer back from the same client object.

You can see the menu without leaving the notebook. `boto3` builds the client's methods from the service definition, so `dir()` on the client is a live table of contents.

In [ ]:
rekognition = boto3.client("rekognition")

[m for m in dir(rekognition) if m.startswith(("detect_", "recognize_", "compare_"))]

Every name in that list is a separate pre-trained model with its own response shape, its own price, and its own documentation page. A few of the ones you just saw:

| Operation | What it answers |
| :--- | :--- |
| `detect_text` | What text appears in this image, and where |
| `detect_labels` | What objects and scenes are in this image (car, road, tree) |
| `detect_faces` | Where are the faces, and what are their attributes |
| `detect_moderation_labels` | Does this image contain explicit or unsafe content |
| `detect_protective_equipment` | Are people wearing hard hats, masks, gloves |
| `compare_faces` | Do these two images show the same person |
| `recognize_celebrities` | Are any public figures in this image |

Today you use exactly one of them, `detect_text`. Knowing the rest exist is the point: when someone asks "can AWS tell me if a claim photo shows a damaged roof," the answer is not "we would have to train something," it is "let me check whether `detect_labels` already does that."

**Reference:** [Rekognition API operations](https://docs.aws.amazon.com/rekognition/latest/APIReference/API_Operations.html)

---
# 2. Look at the folder before you trust it

Start with what is actually on disk. Not what you expect to be on disk.

In [ ]:
all_files = sorted(glob.glob("images/*"))

print(f"{len(all_files)} files in images/")
for path in all_files:
    print(" ", path)

Read that list carefully. Most of those are `.jpg` photos. At least two of them are not photos at all.

That is not a trick, it is what a real intake folder looks like. Somebody dropped a spreadsheet in the wrong place, an upload failed halfway and left a stub. You will handle this properly in Section 7. Right now, just notice it exists, and start with one file you know is good.

In [ ]:
example_path = "images/1c00790f8e592ab2.jpg"

Image.open(example_path)

Look at the photo above and write down, for yourself, everything a human can read in it. The plate number, the state, the registration stickers, the frame slogan. That list is your ground truth for the next few cells. When the API comes back, you want to know what it *should* have found before you see what it *did* find.

---
# 3. The first API call, one line at a time

Here is the call, broken into the pieces that matter.

```python
with open(example_path, "rb") as image_file:
    response = rekognition.detect_text(Image={"Bytes": image_file.read()})
```

- `open(path, "rb")` opens the file in **read-binary** mode. A JPEG is not text. Open it in the default text mode and Python will try to decode the bytes as characters and fail. The `with` block closes the file for you when it exits.
- `image_file.read()` pulls the entire file into memory as a `bytes` object.
- `Image={"Bytes": ...}` is how Rekognition wants the picture. `Image` is a dictionary because there are two ways to hand it an image: `Bytes` for data you have in memory, or `S3Object` for a file already sitting in an S3 bucket. You are using `Bytes`.

Run it, then look at what came back before doing anything with it.

In [ ]:
with open(example_path, "rb") as image_file:
    response = rekognition.detect_text(Image={"Bytes": image_file.read()})

type(response)

**Lesson one: it is a plain Python dictionary.**

There is no special Rekognition result class, no `.text` property, no `.to_dataframe()`. `boto3` parsed the service's JSON response into a `dict`. Everything you do from here is ordinary dictionary and list work, which means the only real question is *what keys are in it*.

In [ ]:
response.keys()

Three keys, and only one of them is your answer:

| Key | What it is |
| :--- | :--- |
| `TextDetections` | The actual result. A list of everything the model read. |
| `TextModelVersion` | Which version of the OCR model ran. Worth logging in production: if your numbers shift next quarter, this tells you whether the model changed underneath you. |
| `ResponseMetadata` | HTTP plumbing. Request ID, status code, retry count. Useful when debugging a failure, noise otherwise. |

Get in the habit of calling `.keys()` on any response object you have not seen before. It is faster than searching the documentation and it tells you what this specific call actually returned, not what it might return.

In [ ]:
print("model version:", response["TextModelVersion"])
print("detections:   ", len(response["TextDetections"]))

Now open exactly one detection. Not all of them. One.

In [ ]:
response["TextDetections"][0]

That single dictionary is the unit Rekognition works in. Every field earns its place:

| Field | Meaning |
| :--- | :--- |
| `DetectedText` | The string the model read. This is the part you probably came for. |
| `Type` | Either `"LINE"` or `"WORD"`. Section 4 is entirely about this. |
| `Confidence` | 0 to 100. How sure the model is. Not a probability, a score. |
| `Id` | An identifier for this detection within this response. |
| `ParentId` | For a `WORD`, the `Id` of the `LINE` it belongs to. Absent on a `LINE`. |
| `Geometry` | Where on the image the text is: a `BoundingBox` and a `Polygon`. |

Notice what is **not** there. There is no `state` field, no `plate_number` field, no `is_a_license_plate` flag. The service reads text. Deciding that one of these strings is a state name is your job, and that is Section 6.

### Geometry is normalized, and that is deliberate

`BoundingBox` gives `Left`, `Top`, `Width`, `Height` as fractions of the image, between 0 and 1, not pixels. A `Left` of `0.38` means "38 percent of the way across," whatever the image resolution happens to be. That way the same coordinates work if you resize the photo.

To turn them into pixels, multiply by the image dimensions. The cell below crops the image down to the first detection's box so you can see that the coordinates really do point at the text.

In [ ]:
box = response["TextDetections"][0]["Geometry"]["BoundingBox"]
image = Image.open(example_path)
width, height = image.size

left = box["Left"] * width
top = box["Top"] * height
right = left + box["Width"] * width
bottom = top + box["Height"] * height

print("detected text:", response["TextDetections"][0]["DetectedText"])
image.crop((left, top, right, bottom))

The crop should contain the string printed above it, and almost nothing else.

That is what makes OCR output more than a list of words: every string carries its location, so you can redact it, highlight it, or filter to a region of the image you care about.

---
# 4. LINE vs WORD, and why it changes your answer

Every piece of text comes back **twice**, once as part of a `LINE` and once as a `WORD`. That is why the detection count is roughly double what you would guess from looking at the photo.

The AWS documentation defines them precisely:

> A word is one or more script characters that are not separated by spaces.
>
> A line is a string of equally spaced words. A line isn't necessarily a complete sentence. For example, a driver's license number is detected as a line.

Print each type separately and the difference becomes obvious.

In [ ]:
for detection in response["TextDetections"]:
    if detection["Type"] == "LINE":
        print(f"{detection['Confidence']:6.2f}%  {detection['DetectedText']!r}")

In [ ]:
for detection in response["TextDetections"]:
    if detection["Type"] == "WORD":
        print(f"{detection['Confidence']:6.2f}%  {detection['DetectedText']!r}")

Compare the two lists.

`LINE` keeps phrases together. A plate number like `VA 5278850` stays as one string, and a frame slogan stays as one readable phrase. `WORD` shatters both into fragments.

**Which one do you want?** It depends entirely on the field you are extracting:

- Looking for a **state name**, either works, because most state names are a single word. But a two-word state such as `New York` or `North Carolina` only survives intact in `LINE`. Search the `WORD` list and you will find `NORTH` and `CAROLINA` separately and match neither.
- Looking for a **plate number**, `LINE` is better, because the number and its state prefix stay together.
- Looking for **individual keywords** such as a manufacturer badge, `WORD` is fine and slightly simpler.

This is the first real design decision in the notebook, and it is invisible unless you actually look at both outputs. Last year's version of this lab used `WORD`, and quietly could never find a two-word state. You will use `LINE`.

`ParentId` is what connects the two views: every `WORD` carries the `Id` of the `LINE` it came from.

In [ ]:
for detection in response["TextDetections"]:
    if detection["Type"] == "WORD" and detection.get("ParentId") == 0:
        print(detection["DetectedText"])

# The words printed here, joined with spaces, reconstruct the text of LINE Id 0.

---
# 5. The documentation is part of the tool

You now know what `detect_text` returns, because you looked. But looking at one response cannot tell you what the service **refuses** to do, what it silently caps, or what options you never passed.

Only the documentation can tell you that, and not knowing it is how people ship jobs that quietly drop data.

Open the API reference and answer these three questions. Do not guess, and do not ask an AI assistant, this is exactly the skill being practiced.

**Reference:** [Rekognition `DetectText` API reference](https://docs.aws.amazon.com/rekognition/latest/APIReference/API_DetectText.html)

### Question 1
The `images/` folder has a `.png` file and a `.csv` file in it. Which file formats does `DetectText` accept? What exception comes back if you send it something else?

### Question 2
Suppose one photo is of a dense page of text with 300 words on it. How many words will `DetectText` return? Find the sentence that says so.

### Question 3
You decide you want to ignore any detection below 80 percent confidence. You could filter in Python after the call. Can the API do it for you instead, and if so, what parameter?

Write your answers in the cell below. The third one is the important one: it is a feature that exists, costs nothing, and is invisible to anyone who never opened the docs.

**Answers:**

1. **Accepted formats.** "The image must be either a .png or .jpeg formatted file." Anything else comes back as an `InvalidImageFormatException`, HTTP 400. Note the consequence for the `.png` sitting in `images/`: PNG is a format the API accepts, so an extension check will happily wave that file through. Whether the file is *actually* a PNG is a separate question the extension cannot answer.

2. **Word cap.** "`DetectText` can detect up to 100 words in an image." Word 101 onward is simply not returned, with no warning and no error. On a dense document you would silently lose most of the page, which is why `DOCUMENT_TEXT_DETECTION`-style operations exist for that job.

3. **Confidence filtering in the API.** Yes. The optional `Filters` parameter takes a `WordFilter` with `MinConfidence`, plus `MinBoundingBoxHeight` and `MinBoundingBoxWidth`. `Filters` also accepts `RegionsOfInterest` to restrict detection to part of the image.

   ```python
   rekognition.detect_text(
       Image={"Bytes": image_bytes},
       Filters={"WordFilter": {"MinConfidence": 80}},
   )
   ```

   Filtering server side means the low-confidence junk never reaches your code at all.

---
# 6. Raw text is not a structured field

Look again at the `LINE` output from Section 4. It contains a state name, some plate numbers, a month, a year, and a tourism slogan, in whatever order the model happened to emit them.

The requested table has a `state` column. So you need code that answers one question: **which of these strings is a US state?**

### Why a list of states, instead of something smarter

A tempting first instinct is to look for a pattern. But there is no pattern. `Virginia` and `Ohio` and `New Mexico` do not share a shape that separates them from `LOVERS` or `AUG` or `OPA`. The only thing that makes `Virginia` a state is that it is on the list of states.

That list is called a **controlled vocabulary**: the fixed set of values a column is allowed to take. It matters for reasons that go well past this notebook:

- **It constrains the output.** The `state` column can only ever contain a real state or nothing. The API cannot invent `Verginia` into your warehouse.
- **It makes rows joinable.** If your state values come from the same list every time, they join cleanly to a population table, a rate table, or a regional dimension. Free text does not join to anything.
- **It makes "not found" meaningful.** A blank means "no state on the list appeared," which is a fact you can count and investigate. If you accepted any string, blank would just mean "the code did not fire."

Building the lookup list is the unglamorous half of most extraction work, and it is the half that decides whether the output is usable.

### Your turn: build the list

Here are the 50 states plus the District of Columbia. Turn them into a Python list.

```
Alabama, Alaska, Arizona, Arkansas, California, Colorado, Connecticut, Delaware,
Florida, Georgia, Hawaii, Idaho, Illinois, Indiana, Iowa, Kansas, Kentucky,
Louisiana, Maine, Maryland, Massachusetts, Michigan, Minnesota, Mississippi,
Missouri, Montana, Nebraska, Nevada, New Hampshire, New Jersey, New Mexico,
New York, North Carolina, North Dakota, Ohio, Oklahoma, Oregon, Pennsylvania,
Rhode Island, South Carolina, South Dakota, Tennessee, Texas, Utah, Vermont,
Virginia, Washington, West Virginia, Wisconsin, Wyoming, District of Columbia
```

**A decision you have to make first.** Go back and compare the two outputs in Section 4. Rekognition returned `VIRGINIA` in all caps on one line and `Virginia.org` in mixed case on another. The list above is in title case. A plain `in` check is case sensitive, so `"Virginia" in "VIRGINIA IS FOR LOVERS"` is `False`.

So you have to normalize. Decide **which case you normalize to**, apply it to *both* sides of the comparison, and be able to say why. There is no single right answer, but there is a wrong answer, which is normalizing only one side.

Keep the list itself in title case. The `state` column should read `New York`, not `NEW YORK`, because the value goes into a report a human will read. Do the case folding at comparison time.

In [ ]:
US_STATES = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado",
    "Connecticut", "Delaware", "Florida", "Georgia", "Hawaii", "Idaho",
    "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", "Louisiana", "Maine",
    "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi",
    "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey",
    "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio",
    "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina",
    "South Dakota", "Tennessee", "Texas", "Utah", "Vermont", "Virginia",
    "Washington", "West Virginia", "Wisconsin", "Wyoming",
    "District of Columbia",
]

print(len(US_STATES), "states in the vocabulary")

Expected: `51 states in the vocabulary`.

### Your turn: write the matcher

Write `find_state`, which takes the list of detected `LINE` strings and returns a state name, or `None` if no state is on any of them.

The approach that works and stays readable:

1. Join all the detected lines into one string, so a state name is found no matter which line it landed on.
2. Normalize the case of that joined string.
3. Walk `US_STATES`, normalizing each state the same way, and return the first one that appears.
4. Return `None` if the loop finishes with no match.

**There is a bug waiting in step 3, and you should find it before you write the code.**

`Virginia` is a substring of `West Virginia`. Your loop returns the **first** state it finds, and the list above is alphabetical, so it checks `Virginia` before it ever reaches `West Virginia`.

Work out what that means for a West Virginia plate. The loop finds `Virginia` inside `WEST VIRGINIA`, returns immediately, and your dataset now says a West Virginia car is registered in Virginia. No error, no warning, just a wrong row.

Write the simple version first and confirm it works on the test below. Then, in a comment, say what you would change to fix the `West Virginia` case. There is a one-line fix that involves the order you iterate the list, and finding it is the exercise.

None of the 31 photos in this batch is a West Virginia plate, so this will not show up in your results today. That is exactly why it is worth writing down now. A bug that your test data cannot trigger is still a bug, and this one ships to production and stays quiet until somebody in Charleston files a claim.

In [ ]:
def find_state(detected_lines):
    """Return the first US state named anywhere in the detected lines, else None."""
    full_text = " ".join(detected_lines).upper()
    for state in US_STATES:
        if state.upper() in full_text:
            return state
    return None

# Case decision: fold both sides to upper case. Plates are frequently printed
# in all caps, so upper case is the form the data most often arrives in, and
# folding both sides means the comparison never depends on how the model chose
# to capitalize a given line.
#
# Substring bug: "Virginia" is inside "West Virginia", and alphabetical order
# checks "Virginia" first, so a West Virginia plate would be labeled "Virginia".
#
# The one-line fix is to check the longest names first, so the most specific
# match wins before a shorter name can shadow it:
#
#     for state in sorted(US_STATES, key=len, reverse=True):
#
# None of the 31 photos here is a West Virginia plate, so both versions produce
# identical output on this batch. The sorted version is still the one to ship,
# because "my test data does not happen to trigger it" is not a fix.

Now test it on the one photo you already have a response for. You know from Section 4 that this plate is a Virginia plate, so you know what the answer has to be.

In [ ]:
example_lines = [
    d["DetectedText"] for d in response["TextDetections"] if d["Type"] == "LINE"
]

print("lines:", example_lines)
print("state:", find_state(example_lines))

Expected: `state: Virginia`.

If you get `None`, your normalization is only applied to one side of the comparison. If you get an error, `find_state` is probably returning nothing because you left the `pass` in.

**Do not continue until this cell prints `Virginia`.** Everything after this point runs `find_state` 62 more times.

---
# 7. Scale it: this part is yours

You have proven the whole thing on one photo: open the file, call the API, pull the `LINE` strings, run `find_state`. Doing it for 31 photos is the same four steps inside a loop.

Two problems appear only at scale, and they are the reason this section is a `TODO` and not a worked example.

### Problem one: the folder has junk in it

You saw it in Section 2. There is a `.csv` and a `.png` sitting in `images/`.

From Section 5 you know `DetectText` accepts `.png` and `.jpeg` only, so an extension check handles the `.csv`. It does **not** handle the `.png`, because PNG is a format the API accepts. An extension is a *claim* a filename makes about its contents, and interrupted uploads, renamed files, and empty stubs break that claim constantly.

So filter in two layers:

1. Keep only paths whose extension is `.jpg`, `.jpeg`, or `.png`.
2. Actually open each survivor with Pillow and call `.verify()` on it. If that raises, the file is not a usable image and gets skipped.

The second layer is the one people forget, and it is the one that takes down a nightly job at 3am.

### Problem two: one bad photo must not kill the batch

A network blip, a throttling error, a corrupt file that got past your filter. If a single failure raises out of the loop, you lose the 20 images you already paid to process. Wrap the API call so a failure is recorded and the loop continues.

In [ ]:
image_paths = []

for path in all_files:
    if not path.lower().endswith((".jpg", ".jpeg", ".png")):
        print(f"skipping (wrong extension): {path}")
        continue
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as e:
        print(f"skipping (unreadable):      {path}  ({e})")
        continue
    image_paths.append(path)

print(f"\n{len(image_paths)} usable images")

Expected: 33 files go in, 31 come out, and **each layer catches a different file**.

- `not_an_image.csv` is stopped by the extension check.
- `ignore_me.png` passes the extension check, because PNG is a format Rekognition accepts. It is caught by the second layer, where Pillow tries to open it and reports `cannot identify image file`. The name says PNG. The bytes disagree.

That second file is the whole point of the section. With only an extension check it would have gone to the API, cost you a call, come back as an `InvalidImageFormatException`, and raised out of your loop somewhere in the middle of the batch. Verifying locally is cheaper and it fails in a place where you can do something about it.

In [ ]:
rekognition_rows = []

for path in image_paths:
    try:
        with open(path, "rb") as image_file:
            result = rekognition.detect_text(Image={"Bytes": image_file.read()})
        lines = [d["DetectedText"] for d in result["TextDetections"] if d["Type"] == "LINE"]
        rekognition_rows.append({
            "image": os.path.basename(path),
            "rekognition_state": find_state(lines) or "Not Found",
            "rekognition_text": " | ".join(lines),
        })
    except Exception as e:
        print(f"\nfailed on {path}: {e}")
        rekognition_rows.append({
            "image": os.path.basename(path),
            "rekognition_state": "API Error",
            "rekognition_text": "",
        })
    print(".", end="", flush=True)

print(f"\ndone: {len(rekognition_rows)} rows")

In [ ]:
rek_df = pd.DataFrame(rekognition_rows)

display(rek_df.head())
rek_df["rekognition_state"].value_counts()

You should now have one row per photo, with a state on most of them.

When this notebook was written, Rekognition found a state on 28 of the 31 photos, with California and Virginia the most common. Your numbers may differ slightly, because these are live models that AWS updates. What should not differ is the shape: one row per image, a real state name or `Not Found` in the state column, and no crash.

A handful of `Not Found` rows is expected and worth a look. Open one of those photos and check whether the plate genuinely has no state name printed on it, or whether the OCR missed it. Those are two completely different problems and only your eyes can tell them apart.

---
# 8. The same job on GCP Vision, one call first

Google Cloud has a service that does the same thing. Comparing them is the point of the activity, but the way you *call* it is a lesson on its own.

With AWS you used `boto3`, an SDK that hides the HTTP. With GCP you are going to call the REST endpoint directly with `requests`. Google publishes a Python client library too, but calling the raw endpoint here is deliberate: an SDK is a convenience wrapper over an HTTP request, and it is worth seeing the request underneath at least once.

### The request

```
POST https://vision.googleapis.com/v1/images:annotate?key=YOUR_API_KEY
```

with a JSON body shaped like this:

```json
{
  "requests": [
    {
      "image":    { "content": "<base64-encoded image bytes>" },
      "features": [ { "type": "TEXT_DETECTION" } ]
    }
  ]
}
```

Three things to notice:

- **`requests` is a list.** Google's endpoint is a batch endpoint by design: you can put several images in one HTTP call. AWS's `detect_text` takes exactly one image per call. That difference matters at volume and it is the kind of thing you only learn by reading the request shape.
- **The image is base64.** JSON cannot carry raw bytes, so binary has to be encoded into text. `base64.b64encode(image_bytes).decode("utf-8")` does it. This makes the request roughly 33 percent larger than the file. That is the cost of sending an image through JSON.
- **`features` is a list too, and this is the same menu you saw in Section 1.** `TEXT_DETECTION` is one of many:

| Feature type | What it answers |
| :--- | :--- |
| `TEXT_DETECTION` | OCR for text in a photo |
| `DOCUMENT_TEXT_DETECTION` | OCR tuned for dense pages and handwriting |
| `LABEL_DETECTION` | What is in the image (car, road, tree) |
| `OBJECT_LOCALIZATION` | What is in the image, plus bounding boxes |
| `LOGO_DETECTION` | Brand logos |
| `FACE_DETECTION` | Faces and their attributes |
| `SAFE_SEARCH_DETECTION` | Explicit content likelihood |
| `WEB_DETECTION` | Where else this image appears on the web |

Because `features` is a list, one call can ask for several at once. Ask for `TEXT_DETECTION` and `LABEL_DETECTION` together and you get both back in one response.

**Reference:** [Cloud Vision feature list](https://cloud.google.com/vision/docs/features-list)

In [ ]:
with open(example_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode("utf-8")

print("original file size:", os.path.getsize(example_path), "bytes")
print("base64 size:       ", len(image_b64), "bytes")

In [ ]:
gcp_response = requests.post(
    "https://vision.googleapis.com/v1/images:annotate",
    params={"key": os.environ["GCP_API_KEY"]},
    json={
        "requests": [
            {
                "image": {"content": image_b64},
                "features": [{"type": "TEXT_DETECTION"}],
            }
        ]
    },
)

print("HTTP status:", gcp_response.status_code)
gcp_response.json().keys()

`requests` does not raise on an HTTP error, it just hands you the response. A `200` means the call succeeded. A `403` almost always means the Cloud Vision API is not enabled on the project the key belongs to, or you used the AI Studio `GOOGLE_API_KEY` instead of `GCP_API_KEY`.

The top level has one key, `responses`, and it is a list, one entry per image you sent. You sent one image, so your answer is at index 0.

In [ ]:
annotation = gcp_response.json()["responses"][0]

annotation.keys()

Two ways of looking at the same OCR result:

- `textAnnotations` is a flat list. **Entry 0 is special**: it is the entire block of text found in the image, newline separated, plus a `locale`. Entries 1 onward are the individual words with their own bounding boxes.
- `fullTextAnnotation` is the same text organized hierarchically, page to block to paragraph to word to symbol. You would reach for this when layout matters, for example reading a form.

Look at entry 0 and entry 1 side by side.

In [ ]:
text_annotations = annotation["textAnnotations"]

print("number of entries:", len(text_annotations))
print()
print("--- entry 0 (the whole block) ---")
print(json.dumps(text_annotations[0], indent=1)[:600])
print()
print("--- entry 1 (one word) ---")
print(json.dumps(text_annotations[1], indent=1)[:400])

In [ ]:
print(text_annotations[0]["description"])

**Compare this to the AWS output.** Both providers read the same photo, but the shapes are not the same:

| | AWS Rekognition | GCP Vision |
| :--- | :--- | :--- |
| Result key | `TextDetections` | `textAnnotations` |
| Granularity | Every item tagged `LINE` or `WORD` | Entry 0 is everything, entries 1+ are words |
| Line separation | The `Type` field | `\n` inside `description` |
| Per-item confidence | `Confidence`, 0 to 100 | Not returned by `TEXT_DETECTION` |
| Coordinates | Normalized 0 to 1 | Absolute pixels |

That last row is a genuine trap: an AWS bounding box and a GCP bounding box are not interchangeable numbers even though both are called a bounding box.

For your purposes the useful consequence is small: to get a list of lines out of GCP, split entry 0's `description` on `\n`. Then `find_state` works on it unchanged, because `find_state` never cared which cloud produced the text.

In [ ]:
gcp_lines = text_annotations[0]["description"].split("\n")

print("lines:", gcp_lines)
print("state:", find_state(gcp_lines))

Expected: `state: Virginia`, the same answer AWS gave, from a completely different response format.

That is the payoff of Section 6. The extraction logic is a property of the *field you want*, not of the vendor. Only the plumbing around it changed.

---
# 9. Now do the GCP batch yourself

You have everything: the request shape, the response shape, the split on `\n`, and `find_state`. Section 7 already showed you the loop pattern. Build the GCP equivalent.

Write a helper function first. You are about to call this 31 times and putting the request-building inline in a loop makes it unreadable.

In [ ]:
def gcp_detect_text(image_bytes):
    """Return the detected text lines from GCP Vision, or [] if none were found."""
    image_b64 = base64.b64encode(image_bytes).decode("utf-8")
    http_response = requests.post(
        "https://vision.googleapis.com/v1/images:annotate",
        params={"key": os.environ["GCP_API_KEY"]},
        json={
            "requests": [
                {
                    "image": {"content": image_b64},
                    "features": [{"type": "TEXT_DETECTION"}],
                }
            ]
        },
    )
    annotations = http_response.json()["responses"][0].get("textAnnotations", [])
    if not annotations:
        return []
    return annotations[0]["description"].split("\n")

In [ ]:
vision_rows = []

for path in image_paths:
    try:
        with open(path, "rb") as image_file:
            lines = gcp_detect_text(image_file.read())
        vision_rows.append({
            "image": os.path.basename(path),
            "vision_state": find_state(lines) or "Not Found",
            "vision_text": " | ".join(lines),
        })
    except Exception as e:
        print(f"\nfailed on {path}: {e}")
        vision_rows.append({
            "image": os.path.basename(path),
            "vision_state": "API Error",
            "vision_text": "",
        })
    print(".", end="", flush=True)

vision_df = pd.DataFrame(vision_rows)
print(f"\ndone: {len(vision_df)} rows")
vision_df["vision_state"].value_counts()

---
# 10. Compare the two clouds, row by row

Two DataFrames, one row per image in each, joined on `image`. What you want out of it is not "which cloud is better" as a slogan, it is a specific, countable claim you could put in a vendor recommendation.

The shape you are building:

| image | rekognition_state | vision_state | agree |
| :--- | :--- | :--- | :--- |
| `1c00790f8e592ab2.jpg` | Virginia | Virginia | True |
| `330ac77b36168d85.jpg` | California | Not Found | False |

Then two numbers: the agreement rate, and the list of rows where they disagreed.

**A warning about the agreement rate.** Agreement is not accuracy. If both providers misread the same plate the same wrong way, they agree and they are both wrong. Agreement tells you how *interchangeable* the two services are, which is genuinely useful when you are deciding whether switching vendors would change your numbers. It does not tell you whether either one is right. Getting to accuracy means looking at the photos yourself, which is the first task in Your Turn.

In [ ]:
comparison = rek_df.merge(vision_df, on="image")
comparison["agree"] = comparison["rekognition_state"] == comparison["vision_state"]

print(f"images compared: {len(comparison)}")
print(f"agreement rate:  {comparison['agree'].mean():.0%}")

display(comparison[["image", "rekognition_state", "vision_state", "agree"]].head(10))

In [ ]:
pd.set_option("display.max_colwidth", None)

disagreements = comparison[~comparison["agree"]]
print(f"{len(disagreements)} disagreements out of {len(comparison)}\n")

for _, row in disagreements.iterrows():
    print(f"--- {row['image']}")
    print(f"  rekognition_state: {row['rekognition_state']}")
    print(f"  vision_state:      {row['vision_state']}")
    print(f"  rekognition read:  {row['rekognition_text']}")
    print(f"  vision read:       {row['vision_text']}")
    print()

### Read the disagreements, do not just count them

Open each disagreeing photo and look at it next to what each provider read. There is usually a specific, explainable cause, and naming that cause is the difference between a finding and a number.

```python
Image.open(f"images/{comparison.loc[3, 'image']}")
```

When this notebook was written, all of the disagreements ran the same direction and had the same cause: Rekognition read `California` correctly and GCP Vision returned a near miss such as `Califorma` or `Valforma` instead. California plates print the state in a stylized script font, and one provider's OCR handled that font better than the other's on these particular photos.

That is a much more useful sentence than "AWS won." It is testable, it explains itself, and it tells the reader exactly when it would and would not apply. If your batch is mostly California plates, the difference matters a lot. If it is mostly block-lettered Virginia plates, the two providers are interchangeable.

Notice also that a one-character OCR error becomes a **total** extraction failure, not a small one. `Califorma` is not in `US_STATES`, so the state comes back `Not Found`. Exact-match lookup against a controlled vocabulary is unforgiving that way, which is worth knowing before you rely on it.

In [ ]:
# Open one of the disagreeing photos and look at it yourself.
# Change the index to walk through them.
disagreeing_images = comparison[~comparison["agree"]]["image"].tolist()
print(disagreeing_images)

Image.open(f"images/{disagreeing_images[0]}") if disagreeing_images else "no disagreements"

---
# 11. The deliverable: a structured dataset

Everything so far has been printed to a notebook. Nobody downstream can use a printed notebook.

Build the final table and write it to disk. This is the actual output of the activity: unstructured pixels went in, a queryable table comes out.

Add one more extracted field while you are here. `make` uses the exact same controlled-vocabulary pattern as `state`, with a different list, which is the point: once you have the pattern, a new field is a new list and nothing else.

**Set your expectations honestly.** A car's manufacturer is only readable if it is physically printed somewhere in the photo, on a dealer frame, a badge, or a bumper sticker. Most of these photos are cropped tight on the plate, so `make` will be empty far more often than `state` is. That is not a bug in your code, it is what the data supports. A field with low coverage is still worth extracting, as long as you report the coverage instead of hiding it.

In [ ]:
CAR_MAKES = [
    "Toyota", "Honda", "Ford", "Chevrolet", "Nissan", "Jeep", "Hyundai", "Kia",
    "Subaru", "Ram", "GMC", "BMW", "Mercedes-Benz", "Lexus", "Audi",
    "Volkswagen", "Tesla", "Acura", "Cadillac", "Chrysler", "Dodge",
    "Infiniti", "Lincoln", "Mazda",
]

print(len(CAR_MAKES), "makes in the vocabulary")

In [ ]:
def find_in_vocabulary(text, vocabulary):
    """Return the first vocabulary term appearing in text, else None."""
    haystack = text.upper()
    for term in vocabulary:
        if term.upper() in haystack:
            return term
    return None


# find_state stays as its own function because it takes a list of lines, which
# is the shape both APIs hand back. find_in_vocabulary is the reusable core.

plates_df = comparison[[
    "image", "rekognition_state", "vision_state", "agree", "rekognition_text",
]].copy()

plates_df["make"] = plates_df["rekognition_text"].apply(
    lambda text: find_in_vocabulary(text, CAR_MAKES) or "Not Found"
)

plates_df = plates_df[[
    "image", "rekognition_state", "vision_state", "agree", "make", "rekognition_text",
]]

total = len(plates_df)
state_found = (plates_df["rekognition_state"] != "Not Found").sum()
make_found = (plates_df["make"] != "Not Found").sum()

print(f"rows:           {total}")
print(f"state coverage: {state_found}/{total} ({state_found / total:.0%})")
print(f"make coverage:  {make_found}/{total} ({make_found / total:.0%})")
print(f"agreement:      {plates_df['agree'].mean():.0%}")

display(plates_df.drop(columns=["rekognition_text"]))

Expect `state` coverage to be high and `make` coverage to be low, in the neighborhood of a fifth of the photos. Report both. A stakeholder who is told "we extracted make" and then finds it empty on 80 percent of rows will stop trusting the rest of the table too.

In [ ]:
plates_df.to_csv("plates_extracted.csv", index=False)

reloaded = pd.read_csv("plates_extracted.csv")
print(f"wrote {len(reloaded)} rows, {len(reloaded.columns)} columns")
print(reloaded.columns.tolist())
reloaded.head()

In [ ]:
plates_df["rekognition_state"].value_counts().plot(
    kind="bar",
    title="License plates by state (AWS Rekognition)",
    xlabel="state",
    ylabel="photos",
)

That chart is the payoff. Thirty-one photographs, which no database can group by and no analyst can filter, became a table with a state column that answers a question in one line.

Nothing in this activity required a training set, a GPU, a model file, or a hyperparameter. That is what "AI as a Service" buys you. What it costs you is control: you cannot fine-tune either model, you cannot fix the California script-font problem, and you inherit whatever biases and blind spots the provider's training data has. That trade is worth it for a narrow, common task somebody else has already solved well. It is a bad trade for anything specific to your business, which is where Day 2's classical models and Day 3's foundation models come back in.

---
# Your Turn

Work in your own copy under `student-work/week6/day4/`.

### 1. Agreement is not accuracy (required)

For every row in `comparison` where the two providers disagreed, open the photo and record by hand which provider, if either, got it right. Build a small table:

| image | rekognition_state | vision_state | truth (your eyes) | who was right |
| :--- | :--- | :--- | :--- | :--- |

Then report each provider's **accuracy** on the disagreements, not their agreement rate with each other. Write one sentence stating which provider you would recommend for this workload and what specifically would change your mind.

### 2. Extract the plate number (required)

Add a `plate_number` column. This is harder than `state` and the difficulty is the lesson: there is no controlled vocabulary of valid plate numbers, so you cannot look it up, you have to describe its shape.

Look at the `rekognition_text` column and characterize what a plate number looks like: how many characters, which mix of letters and digits, and what else in the photo looks similar to it. Registration stickers are the trap, `VA 5278850` and `VA 4713476` are both on the Virginia photo and only one of them is the plate.

Write a version that is right most of the time, then measure how often it is right. "Most of the time" plus a measured number is a real result. Do not chase perfection.

### 3. Confidence gating (required)

Rekognition returns a `Confidence` on every detection. Rebuild the AWS loop keeping only detections above 90 percent confidence, then compare the state column to your original run.

- How many states did you lose by filtering?
- Did you gain any correct answers by dropping a bad read?
- From Section 5 Question 3, is there a way to apply this filter without changing your Python at all?

Then answer the design question: an intake pipeline can auto-accept high-confidence reads and route the rest to a human. Where would you set that line, and what does it cost you if you set it too high or too low?

### Stretch goals

- Ask GCP for `TEXT_DETECTION` and `LABEL_DETECTION` in the same call, since `features` is a list. What labels come back on a vehicle photo, and could they give you the car's color or body type without any OCR at all?
- Time both batches with `time.perf_counter()` and report per-image latency for each provider. Then look up both services' pricing pages and estimate the cost of running this job over 100,000 photos on each cloud. Latency and cost belong in a vendor recommendation alongside accuracy.

---
## What you did

- Discovered a managed AI service's full menu of operations from the client object, not from memory.
- Called `detect_text` on one photo and navigated an unfamiliar response dictionary key by key.
- Used the API reference to find limits and options that no amount of running the code would have revealed.
- Distinguished `LINE` from `WORD` output and picked the one your extraction actually needed.
- Built a controlled vocabulary and used it to turn free OCR text into a constrained, joinable field.
- Scaled a proven single-image pipeline to a whole folder, with defensive filtering and failure isolation.
- Reproduced the same extraction on a second cloud with a different SDK, protocol, and response shape.
- Compared both providers row by row, explained the disagreements, and wrote out a structured dataset.

**Next:** [Activity 2](./Activity_2_Text_AI_AWS_vs_GCP.ipynb) runs the same two-cloud comparison on text instead of images, where the two providers do not even return the same *kind* of answer.